In [87]:
import kagglehub
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")

df = pd.read_csv(path+'/Crop_recommendation.csv')
df.head()

Using Colab cache for faster access to the 'crop-recommendation-dataset' dataset.


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [88]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

Using GPU

In [89]:
dev = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(dev)

cuda


List all the column names

In [90]:
df.columns.tolist()

['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']

Train Test Split

In [91]:
from sklearn.model_selection import train_test_split

X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Standardize the data

In [92]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Label Encoding

In [93]:
from sklearn.preprocessing import LabelEncoder
labels = y.unique()
labeler = LabelEncoder()
y_train = labeler.fit_transform(y_train)
y_test = labeler.transform(y_test)
labels.size

22

Converting this pandas df into tensors

In [94]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32,
    device=dev
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32,
    device=dev
)

y_train = torch.tensor(
    y_train,
    dtype=torch.long,
    device=dev
)

y_test = torch.tensor(
    y_test,
    dtype=torch.long,
    device=dev
)

Creating the MLP

In [95]:
class SoilClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(7, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 22)
        )

    def forward(self, x):
        return self.network(x)

In [96]:
model = SoilClassifier().to(dev)

output = model(X_train)

print(output.shape)

torch.Size([1760, 22])


Creating the optimizer

In [97]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

Training loop with 1000 epochs

In [98]:
epochs = 1000

for epoch in range(epochs):

    # Forward pass
    logits = model(X_train)

    # Calculate loss
    loss = loss_fn(logits, y_train)

    # Backpropagation
    optimizer.zero_grad()

    loss.backward()

    # Update parameters
    optimizer.step()

    if (epoch + 1) % 100 == 0:

        predictions = logits.argmax(dim=1)

        accuracy = (
            predictions == y_train
        ).float().mean()

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {loss.item():.4f} "
            f"Accuracy: {accuracy.item()*100:.2f}%"
        )

Epoch [100/1000] Loss: 2.2923 Accuracy: 41.25%
Epoch [200/1000] Loss: 0.9567 Accuracy: 80.45%
Epoch [300/1000] Loss: 0.4060 Accuracy: 92.90%
Epoch [400/1000] Loss: 0.2209 Accuracy: 96.02%
Epoch [500/1000] Loss: 0.1406 Accuracy: 97.22%
Epoch [600/1000] Loss: 0.0986 Accuracy: 98.01%
Epoch [700/1000] Loss: 0.0742 Accuracy: 98.41%
Epoch [800/1000] Loss: 0.0584 Accuracy: 98.86%
Epoch [900/1000] Loss: 0.0473 Accuracy: 98.98%
Epoch [1000/1000] Loss: 0.0388 Accuracy: 99.20%


Test Accuracy

In [99]:
model.eval()

with torch.no_grad():

    test_logits = model(X_test)

    test_predictions = test_logits.argmax(dim=1)

    accuracy = (
        test_predictions == y_test
    ).float().mean()

print("Test Accuracy:", accuracy.item() * 100)

Test Accuracy: 97.04545140266418
